In [13]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv('Telco_Customer_Churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
# --- PENANGANAN ANOMALI DATA (MISSING VALUES TERSEMBUNYI) ---

# Mendeteksi baris yang berisi spasi kosong (" ") alih-alih nilai kosong (NaN)
empty_spaces = len(df[df['TotalCharges']== " "])
print(f"[check] {empty_spaces}")

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'],errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

#SANITY CHECK (PEMERIKSAAN KEAMANAN) mengenai nilai kosong dan apakah sudah berbentuk numerik
assert df['TotalCharges'].isnull().sum() == 0,"masih ada nilai NaN di total charges"
assert pd.api.types.is_numeric_dtype(df['TotalCharges']),"TotalCharges belum menjadi numerik"

df.drop('customerID', axis=1 , inplace=True)

[check] 11


In [ ]:
# 1. Menyimpan jumlah kolom awal untuk memantau seberapa banyak dimensi data membengkak nanti
cols_before = df.shape[1]

# --- LABEL ENCODING (Kategori Biner) ---

# 2. mengelompokkan kolom yang isinya hanya yes and no  lalu diubah menjadi angka 1 dan 0
yes_no_columns = ['Partner','Dependents','PhoneService','PaperlessBilling','Churn']
for col in yes_no_columns:
    df[col] = df[col].map({'Yes':1,'No':0})

# 3. melakukan hal yang samma untuk gender
df['gender'] = df['gender'].map({'Male':1,'Female':0})

# 4. mengelompokkan kolom yang berisi lebih dari 2 kategori dan memecahnya menjadi kolom dengan kategori 1 dan 0
categorical_cols = ['MultipleLines','InternetService','OnlineSecurity','OnlineBackup',
                    'DeviceProtection','TechSupport','StreamingTV','StreamingMovies',
                    'Contract','PaymentMethod']

df = pd.get_dummies(df, columns=categorical_cols,drop_first=True)

# 5. Memastikan kompatibilitas tipe data
for col in df.columns:
    if df[col].dtype == bool:
        df[col] = df[col].astype(int)

#SANITY CHECK (PEMERIKSAAN KEAMANAN)  mengenai apakah masih ada tipe object seperti string setelah encoding
print(f"[Check] Kolom bertambah dari {cols_before} menjadi {df.shape[1]} setelah One-Hot Encoding.")

object_cols = df.select_dtypes(include=['object']).columns
assert len(object_cols) == 0, f"masih ada kolom berbentuk teks:{list(object_cols)}"

[Check] Kolom bertambah dari 20 menjadi 31 setelah One-Hot Encoding.


In [ ]:
# --- NORMALISASI DATA NUMERIK (SCALING) ---
# Mengubah rentang angka yang timpang (tenure puluhan, charges ribuan) 
# menjadi seragam di antara 0.0 hingga 1.0 agar AI tidak bias saat belajar.
scaler = MinMaxScaler()
num_cols = ['tenure','MonthlyCharges','TotalCharges']

# --- SANITY CHECK (INSPEKSI HASIL SCALING) ---
# Memastikan seluruh nilai baru hasil transformasi berada pas di batas aman (0-1).
df[num_cols] = scaler.fit_transform(df[num_cols])

for col in num_cols:
    max_val = df[col].max()
    min_val = df[col].min()
    print(f"[Check] {col} - Min: {min_val}, Max: {max_val}")
    assert max_val <= 1.0001, f"Peringatan: Nilai max {col} melebihi 1 setelah scaling!"
    assert min_val >= -0.0001, f"Peringatan: Nilai min {col} di bawah 0 setelah scaling!"

[Check] tenure - Min: 0.0, Max: 1.0
[Check] MonthlyCharges - Min: 0.0, Max: 0.9999999999999999
[Check] TotalCharges - Min: 0.0, Max: 1.0


In [ ]:
# --- FINAL CHECK (VERIFIKASI TERAKHIR) ---

# 1. Memastikan kolom target 'Churn' tidak terhapus secara tidak sengaja.
assert 'Churn' in df.columns,"kolom 'Churn' hilang dari dataframe"

# 2. Memastikan seluruh dataset sudah benar-benar 100% bersih dari nilai kosong (NaN)
total_missing =df.isnull().sum().sum()
assert total_missing == 0, f"Kritis: Ditemukan {total_missing} nilai NaN di dataset akhir!"

print("[Sukses] Semua sanity check lolos! Data siap digunakan untuk RL.")

# --- MENYIMPAN HASIL AKHIR ---
df.to_csv('clean_telco_data.csv', index=False)

[Sukses] Semua sanity check lolos! Data siap digunakan untuk RL.


In [ ]:
# --- DATA SPLIT (TRAIN AND TEST) RATIO 8:2 ---
from sklearn.model_selection import train_test_split
df_clean = pd.read_csv('clean_telco_data.csv')

train_df, test_df = train_test_split(
    df_clean, 
    test_size=0.2, 
    random_state=42, 
    stratify=df_clean['Churn']
)

train_df.to_csv('train_telco.csv', index=False)
test_df.to_csv('test_telco.csv', index=False)

print(f"[Sukses] Data Latihan (Train) : {len(train_df)} baris tersimpan di 'train_telco.csv'")
print(f"[Sukses] Data Ujian (Test)    : {len(test_df)} baris tersimpan di 'test_telco.csv'")

[Sukses] Data Latihan (Train) : 5634 baris tersimpan di 'train_telco.csv'
[Sukses] Data Ujian (Test)    : 1409 baris tersimpan di 'test_telco.csv'
